In [1]:
import sys
from pathlib import Path
import os
import tifffile as tiff
import numpy as np

project_root = Path().resolve().parent
sys.path.append(str(project_root / "src"))

In [24]:
## To reload

import importlib
import pinn.movie as movie
importlib.reload(movie)

<module 'pinn.movie' from '/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/src/pinn/movie.py'>

In [27]:
## INPUT FILES

import mrcfile
import os
import numpy as np
from flax.training import checkpoints

# ==== Parameters =====
lambda_1 = 100000
lambda_2 = 1
lambda_3 = 100000
lambda_4 = 0.01
step = 10000
hidden_dim = 128
phase = 2
axis = "x"

# ==== Parameters for Golgi =====
data_id = "czii_27042022"
shape = "golgi_01"
slices = list(range(40, 81, 10))
expand_xy = None

# # ==== Parameters for Mito =====
# data_id = "mendelsohn_2021"
# shape = "mito_01"
# slices = range(100, 301, 50)
# expand_xy = 282/360

# ==== Load checkpoint data =====
checkpoint_dir = f"../outputs/logs/{data_id}/{shape}_{hidden_dim}/phase{phase}_{lambda_1}_{lambda_2}_{lambda_3}_{lambda_4}"
checkpoint_path = os.path.abspath(f"{checkpoint_dir}/checkpoint_{step}")
checkpoint_data = checkpoints.restore_checkpoint(ckpt_dir=checkpoint_path, target=None)

# ===== Load cryoET data =====
if shape == "golgi_01":
    mrc_file_path = f"../data/experimental/vol/formatted/{data_id}/{shape}.mrc"
elif shape == "mito_01":
    mrc_file_path = f"../data/experimental/vol/raw/{data_id}/{shape}.mrc"
with mrcfile.open(mrc_file_path, permissive=True) as mrc:
    cryoET_data = mrc.data
grid_size = cryoET_data.shape[0]

# ===== Load VTK data =====
vtk_path = os.path.abspath(f"{checkpoint_dir}/verts_10000.vtk")

In [29]:
## MAKE MOVIE

from pinn.movie import make_cryoet_phase_gif

# output_path = "../outputs/movie/golgi_raw_to_phase.gif"
output_path = "../outputs/movie/mito_raw_to_phase.gif"

make_cryoet_phase_gif(
    cryoet=cryoET_data,
    checkpoint=checkpoint_data,
    output_path=output_path,
    axis=axis,
    hidden_dim=hidden_dim,
    expand_xy=expand_xy,
    voxel_scale=(1.0, 1.0, 1.0),

    # Animation
    # start_slice=40,
    # end_slice=90,
    slice_step=1,
    # fps=15,
    fps=30,
    transition_pause_seconds=0.5,
    ending_pause_seconds=1.0,

    # Phase-field appearance
    phase_alpha=0.35,

    # Image appearance
    percentile_range=(1.0, 99.5),
    figure_size=(6, 6),
    dpi=120,

    # Computation
    batch=4096,
    run_on_cpu=False,

    # Labels
    show_text=True,
)

Computing 360 phase-field slices...
Phase slice 360/360
Rendering GIF frames...
Overlay frame 360/360
Saved GIF to:
/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/outputs/movie/mito_raw_to_phase.gif
